In [17]:
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt

from diffrax import MultiTerm, WeaklyDiagonalControlTerm, ODETerm, AbstractBrownianPath, SpaceTimeLevyArea, diffeqsolve, SPaRK, SaveAt, SABRController, Euler
from test.helpers import SDE, simple_sde_order

from drawing_and_evaluating import (
    constant_step_strong_order,
    draw_order_multiple,
    plot_sol_general,
)

jax.config.update("jax_enable_x64", True)


# Define the SABR model
def drift(t, y, args):
    return jnp.array([0., -0.5], dtype=y.dtype)

def diffusion(t, y, args):
    v = y[1]
    sigma = jnp.exp(v)
    return jnp.array([sigma, 1.], dtype=y.dtype)

def get_terms(bm: AbstractBrownianPath):
    return MultiTerm(ODETerm(drift), WeaklyDiagonalControlTerm(diffusion, bm))

y0 = jnp.array([0., 0.], dtype=jnp.float64)
t0, t1 = 0., 3.
sabr_sde = SDE(get_terms, None, y0, t0, t1, (2,))

In [18]:
bm_key = jr.key(8)
terms_sabr = get_terms(sabr_sde.get_bm(bm_key, SpaceTimeLevyArea, 2**-8))
# step_ts = jnp.array([t0, 1.17, t1], dtype=jnp.float64)
controller = SABRController(ctol=10000., dtmin=2**-8, dtmax=0.25)
sol = diffeqsolve(terms_sabr, Euler(), t0, t1, 0.1, y0, saveat=SaveAt(steps=True))
# plot_sol_general(sol)
print(sol.ts.shape)
print(sol.ys.shape)

(4096,)
(4096, 2)


In [ ]:
num_samples = 1000
key = jr.key(6)
keys = jr.split(key, num_samples)

sabr_const = constant_step_strong_order(keys, sabr_sde, SPaRK(), (1, 7))

def sabr_strong_order(keys, sde, solver, levels):
    save_ts = jnp.linspace(sde.t0, sde.t1, 65)
    def get_sabr_controller(level: int):
        return None, SABRController(ctol=2**-level, dtmin=2**-12, dtmax=1., step_ts=save_ts)